# Requêtes sources du One Pager

Une section par KPI (numérotation de `docs/notes_et_questions.md`).

- **Requête retenue** : détail au grain fin (une ligne par ligne de production, par batch...), c'est elle qui alimentera les notebooks puis Power BI.
- **Contrôle** : donne la valeur du KPI à date du jour, uniquement pour vérifier la requête retenue.

| KPI | Statut |
|---|---|
| D1 – Data digitalization | ✅ validé (14) |
| D2 – Metamodels | 🟡 en test |
| Autres KPI | requêtes initiales, à reprendre une par une |

## D1 – Data digitalization ✅
Lignes ayant plus de 180 batchs non supprimés (tous statuts, cf. PO1). Date = `created_at` du 181e batch.

In [0]:
%sql
WITH batches_numerotes AS (
    SELECT
        pl.id_plant_production_line AS production_line_id,
        pl.common_code              AS production_line_code,
        pl.name                     AS production_line_name,
        b.id_batch,
        b.created_at,
        ROW_NUMBER() OVER (
            PARTITION BY pl.id_plant_production_line
            ORDER BY b.created_at, b.id_batch
        ) AS rang_batch
    FROM ext_mal_psql_maite_vision_board_prd.public.batches b
    INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
        ON pl.id_plant_production_line = b.production_line
    WHERE b.deleted = false
)
SELECT
    production_line_id,
    production_line_code,
    production_line_name,
    to_date(created_at) AS digitalization_date
FROM batches_numerotes
WHERE rang_batch = 181
ORDER BY digitalization_date;

In [0]:
%sql
SELECT COUNT(*) AS kpi_data_digitalization
FROM (
    SELECT pl.id_plant_production_line
    FROM ext_mal_psql_maite_vision_board_prd.public.batches b
    INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
        ON pl.id_plant_production_line = b.production_line
    WHERE b.deleted = false
    GROUP BY pl.id_plant_production_line
    HAVING COUNT(b.id_batch) > 180
) lignes_digitalisees;

## D2 – Metamodels 🟡
Provisoire : lignes ayant au moins 1 batch non supprimé, hors PI6 (cf. DE1). Date = `created_at` du 1er batch.

In [0]:
%sql
SELECT
    pl.id_plant_production_line AS production_line_id,
    pl.common_code              AS production_line_code,
    pl.name                     AS production_line_name,
    to_date(MIN(b.created_at))  AS first_batch_date,
    COUNT(b.id_batch)           AS nb_batches
FROM ext_mal_psql_maite_vision_board_prd.public.batches b
INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
    ON pl.id_plant_production_line = b.production_line
WHERE b.deleted = false
  AND pl.common_code <> 'PI6'
GROUP BY
    pl.id_plant_production_line,
    pl.common_code,
    pl.name
ORDER BY first_batch_date;

In [0]:
%sql
SELECT COUNT(DISTINCT pl.id_plant_production_line) AS kpi_metamodels
FROM ext_mal_psql_maite_vision_board_prd.public.batches b
INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
    ON pl.id_plant_production_line = b.production_line
WHERE b.deleted = false
  AND pl.common_code <> 'PI6';

## Requêtes initiales des autres KPI
A reprendre KPI par KPI (même méthode : requête retenue + contrôle).

In [0]:
%sql
------AI Recommandations
SELECT COUNT(*) AS NbreSites 
FROM (SELECT DISTINCT 
    PL.name
FROM ext_mal_psql_maite_vision_board_prd.public.plants_production_lines PL
INNER JOIN ext_mal_psql_maite_vision_board_prd.public.batches B
    ON PL.id_plant_production_line = B.production_line
WHERE B.deleted = FALSE AND PL.common_code IN ('RO1','PO1','NG2','PR1','ST2','NG1','BU1')
);

In [0]:
%sql
------Time Tracking et Production Notes
SELECT COUNT(DISTINCT PL.id_plant_production_line) AS NbreSites
FROM ext_mal_psql_maite_vision_board_prd.public.plants_production_lines PL
INNER JOIN ext_mal_psql_maite_vision_board_prd.public.batches B 
    ON PL.id_plant_production_line = B.production_line
INNER JOIN ext_mal_psql_maite_vision_board_prd.public.batches_notes BN 
    ON B.id_batch = BN.batch
INNER JOIN mal_maite_common_prd.gold.trs_metadata MD 
    ON (
        CASE 
            WHEN MD.prd_line = 'burton1' THEN 'BT1'
            WHEN MD.prd_line = 'rouen1' THEN 'RO1'
            WHEN MD.prd_line = 'polisy1' THEN 'PO1'
            WHEN MD.prd_line = 'bolelemi1' THEN 'BO1'
            WHEN MD.prd_line = 'strasbourg1' THEN 'ST1'
            WHEN MD.prd_line = 'strasbourg2' THEN 'ST2'
            WHEN MD.prd_line = 'nogent1' THEN 'NG1'
            WHEN MD.prd_line = 'nogent2' THEN 'NG2'
            WHEN MD.prd_line = 'NYMBURK1' THEN 'NY1' 
            WHEN MD.prd_line = 'prouvy1' THEN 'PR1'
            WHEN MD.prd_line = 'buzau1' THEN 'BU1'
            ---WHEN MD.prd_line = 'pithiviers6' THEN 'PI6'
            WHEN MD.prd_line = 'burton2' THEN 'BT2'
            WHEN MD.prd_line = 'HODONICE1' THEN 'HO1'
            WHEN MD.prd_line = 'KROMERIZ1' THEN 'KR1'
            ELSE MD.prd_line
        END
    ) = PL.common_code
WHERE B.deleted = FALSE;

In [0]:
%sql
------Maltimize unique users
SELECT * FROM ext_mal_psql_maite_vision_board_prd.public.profiles_users
WHERE deleted = 'false';

### U2 – Dashboards PowerBI unique users (requête initiale)
Pas encore de requête : source non disponible.

In [0]:
%sql
------Uncaptured Aveva data
SELECT COUNT(flg_echec) 
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE succes_echec = 'ECHEC'
      AND flg_auto = 1
      AND Id_alerting = 1;

In [0]:
%sql
------Missing manual entries
SELECT SUM(flg_echec)
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE flg_echec = 1 
AND has_manual_value = 1 
AND has_auto_value = 0;

In [0]:
%sql
------Missing auto entries
SELECT SUM(flg_echec)
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE flg_echec = 1 
AND has_manual_value = 0 
AND has_auto_value = 1;

In [0]:
%sql
------Missing auto & manual entries
SELECT SUM(flg_echec)
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE flg_echec = 1 
AND has_manual_value = 1
AND has_auto_value = 1;

In [0]:
%sql
------Total recommandations echec ou succes
SELECT 
    SUM(
        COALESCE(flg_echec, 0) 
        + 
        COALESCE(flg_succes, 0)
    ) AS reco_total
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`;